In [1]:
"""
find_best_demo_clip.py — Trigger-Sense
Scans the AudioSet test set and ranks clips by how well they demonstrate
distinct, temporally-separated event boundaries.

Outputs a ranked shortlist, then plots the top candidates so you can
choose one for Figure 4.5.
"""

import json
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

CLASSES = ["Scream","Shout","Crying","Explosion","Gunshot","Glass","Siren","Alarm"]
SR, N_MELS, TIME_FRAMES = 16000, 128, 128
HOP = 10.0 / 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BEST_THR = {"Scream":0.82,"Shout":0.81,"Crying":0.92,"Explosion":0.75,
            "Gunshot":0.61,"Glass":0.82,"Siren":0.80,"Alarm":0.62}

IDX = json.load(open("audioset_index.json"))


class CRNN_v3(nn.Module):
    def __init__(self, n, sed=True):
        super().__init__()
        self.sed = sed
        def blk(i, o, pool=(2,2)):
            return nn.Sequential(
                nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.Conv2d(o,o,3,padding=1), nn.BatchNorm2d(o), nn.ReLU(),
                nn.MaxPool2d(pool), nn.Dropout2d(0.1))
        self.cnn  = nn.Sequential(blk(1,32), blk(32,64), blk(64,128,(2,1)))
        self.lstm = nn.LSTM(128*16, 128, batch_first=True,
                            bidirectional=True, num_layers=2, dropout=0.2)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(256, n)
    def forward(self, x):
        x = self.cnn(x); b,c,f,t = x.size()
        x = x.permute(0,3,1,2).contiguous().view(b,t,c*f)
        x,_ = self.lstm(x); x = self.drop(x)
        return self.fc(x) if self.sed else self.fc(x.mean(1))


model = CRNN_v3(8).to(DEVICE)
model.load_state_dict(torch.load("best_audioset_sed_v3.pth", map_location=DEVICE))
model.eval()


def run_sed(ytid):
    y, _ = librosa.load(IDX[ytid], sr=SR, duration=10.0)
    if len(y) < 1600:
        return None, None
    mel_raw = librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS))
    mel = (mel_raw - mel_raw.mean()) / (mel_raw.std() + 1e-6)
    mel = (np.pad(mel, ((0,0),(0,TIME_FRAMES-mel.shape[1])))
           if mel.shape[1] < TIME_FRAMES else mel[:, :TIME_FRAMES])
    x = torch.tensor(mel[np.newaxis, np.newaxis], dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        probs = torch.sigmoid(model(x))[0].cpu().numpy()
    return probs, mel_raw


def extract_events(probs, min_frames=1):
    """Return list of (class, start_s, end_s, peak_score)."""
    events = []
    for j, cls in enumerate(CLASSES):
        active = probs[:, j] > BEST_THR[cls]
        inev, start, scores = False, 0.0, []
        for t, a in enumerate(active):
            if a and not inev:
                start, inev, scores = t*HOP, True, [probs[t, j]]
            elif a and inev:
                scores.append(probs[t, j])
            elif not a and inev:
                if (t*HOP - start) >= min_frames*HOP:
                    events.append((cls, start, t*HOP, float(max(scores))))
                inev = False
        if inev:
            events.append((cls, start, 10.0, float(max(scores))))
    return sorted(events, key=lambda e: e[1])


def score_clip(probs, events, true_classes):
    """
    Higher score = better demonstration of distinct temporal boundaries.

    Rewards:  multiple distinct events, correct classes, bounded duration,
              clear silence between events, high peak confidence
    Penalises: events spanning nearly the whole clip (no visible boundary)
    """
    if len(events) < 2:
        return -1  # need at least two events to show separation

    detected = {e[0] for e in events}
    if not detected & set(true_classes):
        return -1  # nothing correct detected

    score = 0.0

    # reward each event that is clearly bounded (not spanning whole clip)
    for cls, s, e, peak in events:
        dur = e - s
        if 0.3 <= dur <= 5.0:          # bounded, visible on a 10s axis
            score += 2.0
            if s > 0.2 and e < 9.8:    # both edges visible, not clipped
                score += 1.5
        elif dur > 8.0:
            score -= 2.0               # spans clip — no visible boundary
        if cls in true_classes:
            score += 1.0
        score += peak                  # prefer confident detections

    # reward temporal gaps between events (visible separation)
    for i in range(len(events) - 1):
        gap = events[i+1][1] - events[i][2]
        if gap > 0.3:
            score += 2.0

    # reward distinct classes rather than repeats of one
    score += 1.5 * len(detected)

    return score


# ─── scan ──────────────────────────────────────────────────────
df = pd.read_csv("audioset_v2_test.csv")
df["ytid"] = df["ytid"].str.strip()
df = df[df["ytid"].isin(IDX)].reset_index(drop=True)

candidates = []
print(f"Scanning {len(df)} test clips...")

for i, row in df.iterrows():
    true_classes = [c for c in CLASSES if row[c] == 1]
    if not true_classes:
        continue
    try:
        probs, mel_raw = run_sed(row["ytid"])
        if probs is None:
            continue
        events = extract_events(probs)
        s = score_clip(probs, events, true_classes)
        if s > 0:
            candidates.append({
                "ytid": row["ytid"], "score": s,
                "true": true_classes, "events": events, "probs": probs,
            })
    except Exception:
        continue

    if (i + 1) % 200 == 0:
        print(f"\r  {i+1}/{len(df)}  ({len(candidates)} candidates)", end="")

candidates.sort(key=lambda c: -c["score"])
print(f"\n\nFound {len(candidates)} usable candidates.\n")

print("=" * 72)
print("TOP 10 CANDIDATES FOR FIGURE 4.5")
print("=" * 72)
for k, c in enumerate(candidates[:10], 1):
    print(f"\n{k}. {c['ytid']}   score={c['score']:.1f}   true={c['true']}")
    for cls, s, e, peak in c["events"]:
        mark = "*" if cls in c["true"] else " "
        print(f"     {mark} {cls:10s} {s:5.2f}s - {e:5.2f}s   (peak {peak:.2f})")

json.dump([{"ytid": c["ytid"], "score": c["score"], "true": c["true"],
            "events": [(cl, round(s,2), round(e,2), round(p,3))
                       for cl, s, e, p in c["events"]]}
           for c in candidates[:20]],
          open("demo_candidates.json", "w"), indent=2)
print("\nSaved shortlist to demo_candidates.json")


# ─── plot the top 5 so you can choose visually ─────────────────
print("\nPlotting top 5 candidates...")
for k, c in enumerate(candidates[:5], 1):
    probs = c["probs"]
    _, mel_raw = run_sed(c["ytid"])
    t_ax = np.arange(probs.shape[0]) * HOP

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), height_ratios=[1.3, 1])
    axes[0].imshow(mel_raw, aspect="auto", origin="lower", cmap="magma",
                   extent=[0, 10, 0, N_MELS])
    axes[0].set_ylabel("Mel bin")
    axes[0].set_title(f"Mel Spectrogram — {c['ytid']}  (true: {c['true']})")

    cols = plt.cm.tab10(np.linspace(0, 1, 8))
    for j, cls in enumerate(CLASSES):
        active = cls in c["true"]
        axes[1].plot(t_ax, probs[:, j], color=cols[j], label=cls,
                     lw=2.4 if active else 1.0, alpha=1.0 if active else 0.25)
        axes[1].axhline(BEST_THR[cls], color=cols[j], ls=":", lw=0.6, alpha=0.3)

    axes[1].set_xlim(0, 10); axes[1].set_ylim(0, 1)
    axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Frame probability")
    axes[1].set_title("Frame-Level SED Output (bold = ground truth)")
    axes[1].legend(ncol=4, fontsize=8); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fname = f"candidate_{k}_{c['ytid']}.png"
    plt.savefig(fname, dpi=180)
    plt.close()
    print(f"  {fname}   (score {c['score']:.1f})")

print("\nReview the candidate_*.png files and pick the clearest one.")
print("Then set that ytid in generate_figures.py -> figure_4_5(demo_ytid=...)")

/tmp/ipykernel_1801828/1273012704.py:52: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("best_audioset_sed_v3.pth", map_location=DEVICE))


FileNotFoundError: [Errno 2] No such file or directory: 'best_audioset_sed_v3.pth'